# Week 5 Lab - Voltage glitching the counting loop

The **same** target loop, reached through the **crowbar** instead of the clock.

**Hardware:** connect the Husky's **Crowbar** SMA output to **either** of the CW313's two SMA connectors - both
tie to the target VCC / shunt line, so it does not matter which. The crowbar pulse reaches the SAM4S VCC over that
coax (not through the 20-pin ribbon), so this cable must be connected for the voltage pass. Based on ChipWhisperer
`fault101` Fault 2_1. The notebook finds your ChipWhisperer install automatically, so it runs from anywhere.

In [ ]:
SCOPETYPE = 'OPENADC'
PLATFORM  = 'CW312_SAM4S'
SS_VER    = 'SS_VER_2_1'

In [ ]:
# Locate the ChipWhisperer install from the installed package, so this notebook
# runs from anywhere - it does not need to sit inside the chipwhisperer tree.
import os, chipwhisperer as cw
CWROOT = os.path.normpath(os.path.join(os.path.dirname(cw.__file__), "..", ".."))
SETUP  = os.path.join(CWROOT, "jupyter", "Setup_Scripts", "Setup_Generic.ipynb")
FW_SRC = os.path.join(CWROOT, "firmware", "mcu", "simpleserial-glitch")
# drive-less, forward-slash root for the %%bash build cell; the cell itself finds
# the right mount prefix (/mnt/c on WSL, /c on Git-Bash, / on macOS/Linux).
CW_BASH = (CWROOT[0].lower() + CWROOT[2:].replace("\\", "/")) if os.name == "nt" else CWROOT.lstrip("/")
print("ChipWhisperer:", CWROOT)

In [ ]:
%run "$SETUP"    # brings up scope, prog, target

In [ ]:
%%bash -s "$PLATFORM" "$SS_VER" "$CW_BASH"
# find the firmware dir under whichever mount this bash uses (WSL /mnt/c, Git-Bash /c, macOS /)
FW=""
for p in "/mnt/$3" "/$3"; do
  [ -d "$p/firmware/mcu/simpleserial-glitch" ] && FW="$p/firmware/mcu/simpleserial-glitch" && break
done
[ -z "$FW" ] && { echo "firmware dir not found for root: $3" >&2; exit 1; }
cd "$FW"
make PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 -j

In [ ]:
fw_path = os.path.join(FW_SRC, "simpleserial-glitch-{}.hex".format(PLATFORM))
cw.program_target(scope, prog, fw_path)
if SS_VER == 'SS_VER_2_1':
    target.reset_comms()

In [ ]:
def reboot_flush():
    scope.io.nrst = False
    time.sleep(0.05)
    scope.io.nrst = "high_z"
    time.sleep(0.05)
    target.flush()

## The code we are glitching

The `'g'` command runs this on the target:

```c
uint8_t glitch_loop(uint8_t* in)
{
    volatile uint16_t i, j;
    volatile uint32_t cnt;
    cnt = 0;
    trigger_high();
    for(i=0; i<50; i++){
        for(j=0; j<50; j++){
            cnt++;
        }
    }
    trigger_low();
    simpleserial_put('r', 4, (uint8_t*)&cnt);
    return (cnt != 2500);
}
```

`volatile` stops the compiler optimising the loop into a constant. `trigger_high()/trigger_low()` bracket the loop: the scope arms on the rising edge and `ext_offset` counts cycles from it. The loop counts to 50x50 = 2500 and sends it back. Any landed fault makes the returned count != 2500.

## Baseline: confirm 2500

With no glitch, the target must answer 2500 before we attack it.

In [ ]:
import struct
target.simpleserial_write("g", bytearray([]))
val = target.simpleserial_read_witherrors('r', 4, glitch_timeout=10)
print(struct.unpack("<I", val['payload'])[0])   # expect 2500

## Set up voltage glitching (crowbar)

`output='enable_only'` holds the crowbar on for a **full clock cycle** per glitch (repeated `scope.glitch.repeat`
cycles) instead of a sub-cycle pulse. On this SAM4S the short `glitch_only` pulse only ever resets or does nothing;
the full-cycle `enable_only` glitch is what actually corrupts the loop. `glitch_hp` / `glitch_lp` enable both
crowbar MOSFETs. In `enable_only` mode `width` and `offset` have no effect - the knobs are `repeat` (pulse length
in cycles) and `ext_offset` (where in the loop the pulse lands).

In [ ]:
if scope._is_husky:
    scope.glitch.enabled = True
    scope.glitch.clk_src = "pll"
else:
    scope.glitch.clk_src = "clkgen"
scope.glitch.output = "enable_only"   # full-cycle crowbar pulse; glitch_only can't corrupt this SAM4S
scope.glitch.trigger_src = "ext_single"
scope.io.glitch_hp = True
scope.io.glitch_lp = True             # both crowbar MOSFETs
if scope._is_husky:
    scope.glitch.enabled = True       # re-enable in case a setting cleared it
print("enabled:", scope.glitch.enabled, "| output:", scope.glitch.output)

## Glitch controller and live map

Same as the clock lab: `gc` keeps the live counters, and the next cell builds the bokeh live map (a `BokehModel`
ipywidget, which is what streams in VS Code). The map plots `ext_offset` (where in the loop the pulse lands)
against `repeat` (pulse length in cycles). Green `+` success, red `x` reset, light-gray dots the swept
background (normal); click a legend entry to hide that layer. Run both before the campaign.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
gc = cw.GlitchController(groups=["success", "reset", "normal"], parameters=["repeat", "ext_offset"])
# Live counts only. gc.display_stats() also shows "<param> setting" sliders, but those stay at 0: this
# lab drives scope.glitch directly and only feeds results back with gc.add(), so gc never moves its own
# parameter sliders. The live map below is the real view of what is swept.
gc.widget_list_groups = [widgets.IntText(value=0, description=g + " count:", disabled=True)
                         for g in gc.groups]
display(*gc.widget_list_groups)

In [ ]:
# Live bokeh map - updates per shot in VS Code.
# output_notebook() loads BokehJS (without it the BokehModel widget stays blank);
# BokehModel wraps the figure as an ipywidget, so stream() pushes points live.
from bokeh.io import output_notebook
from bokeh.plotting import figure
from bokeh.models import ColumnDataSource
from jupyter_bokeh import BokehModel
from IPython.display import display
output_notebook()

src_normal  = ColumnDataSource(dict(x=[], y=[]))
src_reset   = ColumnDataSource(dict(x=[], y=[]))
src_success = ColumnDataSource(dict(x=[], y=[]))

live_p = figure(width=820, height=600, title="Voltage-glitch map",
                x_axis_label="ext_offset", y_axis_label="repeat")
live_p.scatter("x", "y", source=src_normal,  size=3,  color="lightgray",          legend_label="normal")
live_p.scatter("x", "y", source=src_reset,   size=9,  marker="x",     color="red",   legend_label="reset")
live_p.scatter("x", "y", source=src_success, size=12, marker="cross", color="green", legend_label="success")
live_p.legend.click_policy = "hide"    # click a legend entry to show/hide that layer
display(BokehModel(live_p))

_live_src = {"normal": src_normal, "reset": src_reset, "success": src_success}

def live_add(group, ext_offset, repeat):
    _live_src[group].stream(dict(x=[ext_offset], y=[repeat]))

## Campaign - run it for a while and map the band

One experiment cell, same shape as the clock lab, but through the crowbar in `enable_only` mode. It sweeps the
shuffled grid `PASSES` times (or until `MAX_MINUTES`), classifies every shot, and streams success/reset onto the
map above.

Voltage-specific points:

- **`repeat`** is the pulse length in whole clock cycles. Too few = no effect (normal), too many = the rail
  crashes (reset); the success band on this board is **repeat 3-5**. `width` / `offset` do nothing in `enable_only`.
- **`ext_offset`** is swept **across the whole loop (0-3000)**, not just the first few cycles. The loop-skip
  sensitive points recur throughout the 50x50 loop, so the deep sweep is what finds them - a shallow `ext_offset`
  sweep (0-20, like the clock lab) catches almost nothing here.
- **`scope.io.vglitch_reset()`** discharges the crowbar after every shot, or the next shot starts from a
  disturbed rail.

Measured on this board: a few percent up to ~15% of shots land a fault, typically strongest at **repeat 5**
(close to half the shots at that one value), with the faulted count nudged below 2500. The exact rate and best
`repeat` shift run-to-run and board-to-board (SMA cable, shunt, MOSFET spread), so read your own map rather than
expecting a fixed number.

In [ ]:
import struct, random, time

# --- campaign knobs: raise these to run longer / collect more ---
PASSES      = 3     # how many full sweeps of the grid
MAX_MINUTES = 20    # hard wall-clock cap (stops on whichever comes first)
# ----------------------------------------------------------------

# gc.add() keeps the counters/calc; live_add() streams success/reset onto the map.
cw.set_all_log_levels(cw.logging.CRITICAL)
if scope._is_husky:
    scope.glitch.enabled = True
scope.adc.lo_gain_errors_disabled = True
scope.adc.clip_errors_disabled = True
scope.adc.timeout = 0.5

# enable_only: width/offset do nothing. Sweep repeat (pulse length) and ext_offset (position in the loop).
# ext_offset must span the whole loop - the sensitive skip points recur throughout it.
repeats = [3, 4, 5]
exts    = list(range(0, 3000, 3))
grid = [(r, e) for r in repeats for e in exts]

tally = {"success": 0, "reset": 0, "normal": 0}
deadline = time.time() + MAX_MINUTES * 60
t0 = time.time()
n = 0
reboot_flush()
stop = False
for p in range(PASSES):
    random.shuffle(grid)                 # fresh order each pass
    for r, e in grid:
        if time.time() > deadline:
            stop = True; break
        n += 1
        if scope.adc.state:              # trigger still high from a crash
            reboot_flush()
        scope.glitch.repeat, scope.glitch.ext_offset = r, e
        scope.arm()
        target.simpleserial_write("g", bytearray([]))
        ret = scope.capture()
        scope.io.vglitch_reset()         # discharge the crowbar after every shot
        if ret:                          # no trigger -> crashed
            group = "reset"
        else:
            val = target.simpleserial_read_witherrors('r', 4, glitch_timeout=10, timeout=50)
            if not val['valid'] or val['payload'] is None:
                group = "reset"
            else:
                cnt = struct.unpack("<I", val['payload'])[0]
                group = "normal" if cnt == 2500 else "success"
        tally[group] += 1
        gc.add(group, (r, e))
        live_add(group, e, r)
        if group == "reset":
            reboot_flush()
        if n % 500 == 0:
            print(f"{n:6d} shots | {time.time()-t0:4.0f}s | "
                  f"success={tally['success']:4d} reset={tally['reset']:4d} normal={tally['normal']:5d}")
    if stop:
        break

el = time.time() - t0
print(f"\nDONE: {n} shots in {el:.0f}s ({n/max(el,1):.0f}/s), {p+1} pass(es)")
print("tally:", tally, "| success rate: %.2f%%" % (100 * tally['success'] / max(n, 1)))

## Results

The per-setting table, sorted by success rate (ext_offset collapsed) - read the working `repeat` from it. The
live map above already shows where in the loop (ext_offset) the faults land.

In [ ]:
results = gc.calc(ignore_params="ext_offset", sort="success_rate")
results

## See the glitch on the scope - what the knobs do

The capture ADC records a power trace on every `scope.capture()`. With the Husky **MEASURE** SMA
connected to the CW313 (the second coax, alongside the Crowbar one), the crowbar shows up as a
**positive spike** on that trace - so the two voltage knobs become visible:

- **`ext_offset`** moves *where* the spike lands in the loop.
- **`repeat`** sets *how long* the crowbar stays on - the spike widens.

This view needs the **MEASURE** SMA connected (the campaign itself needs only Crowbar). The spike is
visible even where no fault lands, so any `WORK_EXT` in range works for this view - set it near a working
`ext_offset` from the table above.

In [ ]:
%matplotlib inline
import numpy as np, struct
import matplotlib.pyplot as plt

WORK_EXT = 150          # an ext_offset to centre the view on (use a working value from the table)
scope.adc.samples = 1500

def capture(crowbar, ext, rep):
    scope.io.glitch_hp = crowbar
    scope.io.glitch_lp = crowbar
    if scope._is_husky:
        scope.glitch.enabled = True
    scope.glitch.ext_offset, scope.glitch.repeat = ext, rep
    reboot_flush(); scope.arm()
    target.simpleserial_write("g", bytearray([]))
    scope.capture(); scope.io.vglitch_reset()
    val = target.simpleserial_read_witherrors('r', 4, glitch_timeout=10, timeout=50)
    cnt = None if (not val['valid'] or val['payload'] is None) else struct.unpack("<I", val['payload'])[0]
    return np.array(scope.get_last_trace()), cnt

base, _ = capture(False, WORK_EXT, 4)          # no crowbar = clean power trace
g, _    = capture(True,  WORK_EXT, 4)
i = int(np.argmax(np.abs(g - base)))           # where the crowbar lands
lo, hi = max(0, i - 130), min(len(base), i + 130)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
ax1.plot(range(lo, hi), base[lo:hi], color='0.6', lw=1, label='no glitch')
for e in [WORK_EXT - 45, WORK_EXT, WORK_EXT + 45]:
    tr, c = capture(True, e, 4)
    ax1.plot(range(lo, hi), tr[lo:hi], lw=1, label=f'ext_offset={e} (count={c})')
ax1.set_title('ext_offset moves WHERE the crowbar fires (repeat=4)'); ax1.legend(fontsize=8)
ax2.plot(range(lo, hi), base[lo:hi], color='0.6', lw=1, label='no glitch')
for r in [1, 4, 8]:
    tr, c = capture(True, WORK_EXT, r)
    ax2.plot(range(lo, hi), tr[lo:hi], lw=1, label=f'repeat={r} (count={c})')
ax2.set_title('repeat sets HOW LONG the crowbar fires (ext_offset=%d)' % WORK_EXT); ax2.legend(fontsize=8)
ax2.set_xlabel('ADC sample'); plt.tight_layout(); plt.show()

## Submission

Same as the clock lab: working parameters, a count that is **not 2500**, and the working range - reached through the power rail this time.

In [ ]:
scope.dis()
target.dis()